# 01e — SymbTr Turkish Makam: Exploratory Data Analysis

This notebook characterises the Turkish Makam subset (200 MIDI files from SymbTr).
Turkish Makam music uses a microtonal pitch system based on 53-tone equal temperament
(53-TET), where pitches can deviate from the 12-TET semitone grid by multiples of a
*comma* (~22.6 cents). This is the core musical feature that EC-REMI must encode.

This notebook covers:
1. Standard MIDI-level statistics (duration, note density, pitch range, PC entropy)
2. Makam, usul, and form distributions
3. A preview of the 53-TET Koma data from one SymbTr TXT file — the microtonal
   ground truth that will calibrate EC-REMI deviation tokens in Step 5.


In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.midi_utils import (
    load_midi, analyse_midi, get_pitch_class_histogram,
    pitch_class_entropy, piano_roll_plot
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

PC_LABELS = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [2]:
def build_stats_df(midi_dir, meta_df=None, id_col=None):
    """Analyse all MIDI files in midi_dir; optionally merge metadata."""
    midi_files = sorted(midi_dir.glob("*.mid")) + sorted(midi_dir.glob("*.midi"))
    print(f"Analysing {len(midi_files)} MIDI files ...")
    records = [analyse_midi(p) for p in midi_files]
    df = pd.DataFrame(records)
    if "error" in df.columns:
        bad = df["error"].notna().sum()
        if bad:
            print(f"  Warning: {bad} files failed to load.")
        df = df[df["error"].isna()].drop(columns=["error"])
    df["filename"] = [Path(p).name for p in df["path"]]
    if meta_df is not None and id_col is not None:
        df = df.merge(meta_df, left_on="filename", right_on=id_col, how="left")
    return df

In [3]:
def summary_panel(df, tradition_name, save_path):
    """4-panel summary figure: duration, note density, pitch range, PC entropy."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"{tradition_name} — EDA Summary (n={len(df)})", fontsize=13, y=1.01)

    # Duration
    ax = axes[0, 0]
    df["duration_s"].div(60).plot.hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
    ax.axvline(df["duration_s"].mean()/60, color="red", linestyle="--", label=f"mean={df['duration_s'].mean()/60:.1f} min")
    ax.set_xlabel("Duration (minutes)")
    ax.set_title("Duration Distribution")
    ax.legend(fontsize=8)

    # Note density
    ax = axes[0, 1]
    df["note_density"].plot.hist(bins=25, ax=ax, color="seagreen", edgecolor="white")
    ax.axvline(df["note_density"].mean(), color="red", linestyle="--", label=f"mean={df['note_density'].mean():.2f}")
    ax.set_xlabel("Notes per second")
    ax.set_title("Note Density Distribution")
    ax.legend(fontsize=8)

    # Pitch range
    ax = axes[1, 0]
    df["pitch_range"].plot.hist(bins=25, ax=ax, color="darkorange", edgecolor="white")
    ax.axvline(df["pitch_range"].mean(), color="red", linestyle="--", label=f"mean={df['pitch_range'].mean():.1f}")
    ax.set_xlabel("Pitch range (semitones)")
    ax.set_title("Pitch Range Distribution")
    ax.legend(fontsize=8)

    # PC entropy
    ax = axes[1, 1]
    df["pc_entropy"].plot.hist(bins=25, ax=ax, color="mediumpurple", edgecolor="white")
    ax.axvline(df["pc_entropy"].mean(), color="red", linestyle="--", label=f"mean={df['pc_entropy'].mean():.3f}")
    ax.set_xlabel("Pitch Class Entropy (bits)")
    ax.set_title("PC Entropy Distribution\n(Yang & Lerch, 2020)")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")


def mean_pc_histogram_plot(df_midi_paths, tradition_name, save_path):
    """Plot the mean pitch class histogram across all pieces."""
    hists = []
    for p in df_midi_paths:
        pm = load_midi(p)
        if pm:
            hists.append(get_pitch_class_histogram(pm))
    if not hists:
        return
    mean_hist = np.mean(hists, axis=0)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(PC_LABELS, mean_hist, color="steelblue", edgecolor="white")
    ax.set_xlabel("Pitch class")
    ax.set_ylabel("Mean relative frequency")
    ax.set_title(f"{tradition_name} — Mean Pitch Class Histogram (n={len(hists)})")
    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")

In [4]:
MIDI_DIR = PROJECT_ROOT / "data" / "processed" / "turkish_makam" / "midi"
META_CSV = PROJECT_ROOT / "data" / "metadata" / "symbtr_selected.csv"
TXT_DIR  = PROJECT_ROOT / "datasets" / "SymbTr" / "txt"

midi_files = list(MIDI_DIR.glob("*.mid"))
if not midi_files:
    raise FileNotFoundError(
        "No MIDI files found. Run notebook 00e_turkish_makam_prep.ipynb first."
    )
print(f"Found {len(midi_files)} MIDI files.")

meta = pd.read_csv(META_CSV)
print(f"Metadata rows: {len(meta)}")
meta[["makam", "form", "usul", "composer"]].head(5)

Found 200 MIDI files.
Metadata rows: 200


,makam,form,usul,composer
0,rast,seyir,senginsemai,erol bingol
1,neva,seyir,sofyan,sefik gurmeric
2,sababuselik,kupe,aksak,ahmet avni konuk
3,rast,sarki,semai,erol sayan
4,suzidilara,kupe,devrirevanihindi,ahmet avni konuk


## 1. Makam, form, and usul distributions

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

meta["makam"].value_counts().head(20).plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Top 20 Makams")
axes[0].tick_params(axis="x", rotation=60, labelsize=8)
axes[0].set_ylabel("Count")

meta["usul"].value_counts().head(15).plot(
    kind="bar", ax=axes[1], color="seagreen", edgecolor="white")
axes[1].set_title("Top 15 Usuls (rhythmic cycles)")
axes[1].tick_params(axis="x", rotation=60, labelsize=8)

meta["form"].value_counts().head(12).plot(
    kind="bar", ax=axes[2], color="darkorange", edgecolor="white")
axes[2].set_title("Top 12 Forms")
axes[2].tick_params(axis="x", rotation=60, labelsize=8)

plt.suptitle("SymbTr Turkish Makam — Metadata Distribution (n=200)", fontsize=12)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_turkish_metadata_dist.png"), dpi=150)
plt.show()

## 2. Compute statistics

In [6]:
stats = build_stats_df(MIDI_DIR)
print(f"Files analysed: {len(stats)}")
print("\nDescriptive statistics:")
print(stats[["duration_s", "note_count", "note_density", "pitch_range", "pc_entropy"]].describe().round(3))

Analysing 200 MIDI files ...


Files analysed: 200

Descriptive statistics:
       duration_s  note_count  note_density  pitch_range  pc_entropy
count     200.000     200.000       200.000      200.000     200.000
mean      130.483     290.930         2.026       15.755       2.719
std        97.536     232.321         0.870        5.500       0.721
min         0.000       0.000         0.000        0.000      -0.000
25%        63.800      93.000         1.590       14.000       2.734
50%       112.600     267.000         1.981       17.000       2.910
75%       182.147     423.500         2.440       19.000       3.027
max       651.420    1203.000         5.878       29.000       3.365


## 3. Distribution plots

In [7]:
summary_panel(stats, "Turkish Makam (SymbTr)", RESULTS_DIR / "eda_turkish_summary.png")

Saved → results/eda_turkish_summary.png


## 4. Mean pitch class histogram

In [8]:
mean_pc_histogram_plot(stats["path"].tolist(), "Turkish Makam (SymbTr)",
                       RESULTS_DIR / "eda_turkish_pc_histogram.png")

Saved → results/eda_turkish_pc_histogram.png


## 5. PC entropy by makam (top 10)

Different makams have characteristic modal frameworks and may show distinct pitch
class distributions. High-entropy makams use more pitch classes; low-entropy ones
have a strong tonal focus.


In [9]:
stats["filename"] = stats["filename"].astype(str)
# Extract makam from processed filename: symbtr_NNN_makam_form.mid
stats["makam"] = stats["filename"].str.extract(r"symbtr_\d+_([^_]+)_")

entropy_by_makam = (
    stats.groupby("makam")["pc_entropy"]
    .agg(["mean", "std", "count"])
    .sort_values("mean", ascending=False)
)
print("PC Entropy by makam (top 15):")
print(entropy_by_makam.head(15).round(3).to_string())

PC Entropy by makam (top 15):
                 mean    std  count
makam                              
nisabureyn      3.365    NaN      1
vecdidil        3.327    NaN      1
vechisehnaz     3.268    NaN      1
arazbar         3.267    NaN      1
nihavendikebir  3.230    NaN      1
hicazasiran     3.184    NaN      1
bendihisar      3.170    NaN      1
hicazkar        3.167  0.108      4
ruyiirak        3.167    NaN      1
sabaasiran      3.164    NaN      1
gulizar         3.160    NaN      1
acembuselik     3.146    NaN      1
ferahfeza       3.144    NaN      1
sehnaz          3.140    NaN      1
gulzar          3.133    NaN      1


## 6. Microtonal preview — 53-TET Koma data

The SymbTr TXT files encode pitch in 53-TET commas via the `Koma53` column.
This is the only tradition in this study with native microtonal notation in the
source dataset. The distribution of Koma53 values within a makam shows its
characteristic pitch inflections — which standard REMI (12-TET only) cannot represent,
and which EC-REMI deviation tokens are designed to encode.

This analysis directly motivates the EC-REMI design in Step 5.


In [10]:
# Load Koma53 data from the original TXT files for the selected pieces
koma_records = []
for _, row in meta.head(50).iterrows():  # sample 50 to keep analysis fast
    # Original filename (before renaming)
    orig_name = row["filename"].replace(".mid", ".txt")
    txt_path  = TXT_DIR / orig_name
    if not txt_path.exists():
        continue
    try:
        txt_df = pd.read_csv(txt_path, sep="\t", encoding="utf-8")
        # Keep only rows that are actual notes (Koma53 is numeric)
        txt_df = txt_df[pd.to_numeric(txt_df["Koma53"], errors="coerce").notna()].copy()
        txt_df["Koma53"] = pd.to_numeric(txt_df["Koma53"])
        txt_df["makam"]  = row["makam"]
        koma_records.append(txt_df[["Koma53", "NotaAE", "makam"]])
    except Exception:
        continue

if koma_records:
    koma_df = pd.concat(koma_records, ignore_index=True)
    print(f"Total note events with Koma53 data: {len(koma_df):,}")
    print(f"Koma53 range: {koma_df['Koma53'].min():.0f} – {koma_df['Koma53'].max():.0f}")
    print(f"Unique Koma53 values: {koma_df['Koma53'].nunique()}")
    print("\nTop 15 pitch names (Western equivalent):")
    print(koma_df["NotaAE"].value_counts().head(15).to_string())
else:
    print("No TXT files found — check that the SymbTr TXT directory is correct.")

Total note events with Koma53 data: 14,867
Koma53 range: -1 – 380
Unique Koma53 values: 55

Top 15 pitch names (Western equivalent):
NotaAE
D5      2497
C5      1629
E5      1416
G5      1183
A4      1079
B4b1    1060
A5       659
F5#4     657
F5       655
G4       609
Es       506
C5#4     443
B4       325
E5b4     266
B4b4     178


In [11]:
if koma_records and len(koma_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Koma53 distribution (all sampled pieces)
    koma_df["Koma53"].plot.hist(bins=53, ax=axes[0], color="steelblue", edgecolor="white")
    axes[0].set_xlabel("53-TET Koma value")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title("Koma53 Distribution (all sampled makams)\n"
                       "Non-zero = microtonal deviation from 12-TET")
    axes[0].axvline(0, color="red", linestyle="--", alpha=0.5, label="12-TET ref")
    axes[0].legend()

    # Deviation from nearest 12-TET semitone
    # In 53-TET: one semitone = ~4.42 commas. Deviation = Koma53 mod 4.42
    koma_df["deviation_commas"] = koma_df["Koma53"] % (53/12)
    koma_df["deviation_commas"].plot.hist(
        bins=30, ax=axes[1], color="indianred", edgecolor="white")
    axes[1].set_xlabel("Microtonal deviation from nearest 12-TET pitch (commas)")
    axes[1].set_ylabel("Frequency")
    axes[1].set_title("Microtonal Deviation Distribution\n"
                       "Motivates EC-REMI deviation tokens")

    plt.suptitle("SymbTr 53-TET Koma Analysis", fontsize=12)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / "eda_turkish_koma53.png"), dpi=150)
    plt.show()
    print("Saved → results/eda_turkish_koma53.png")

Saved → results/eda_turkish_koma53.png


## 7. Sample piano roll

In [12]:
sample_path = stats["path"].iloc[0]
pm = load_midi(sample_path)
sample_name = Path(sample_path).stem[:60]

fig, ax = plt.subplots(figsize=(14, 4))
piano_roll_plot(pm, ax, time_start=0, time_end=30,
                title=f"Piano roll — {sample_name}")
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_turkish_piano_roll.png"), dpi=150)
plt.show()

## 8. Save summary statistics + cross-tradition preview

In [13]:
stats["tradition"] = "turkish_makam"
stats.to_csv(RESULTS_DIR / "eda_turkish_stats.csv", index=False)
print("Saved → results/eda_turkish_stats.csv")
print(f"\nPC entropy mean : {stats['pc_entropy'].mean():.3f} bits")
print(f"Note density mean: {stats['note_density'].mean():.2f} notes/s")

Saved → results/eda_turkish_stats.csv

PC entropy mean : 2.719 bits
Note density mean: 2.03 notes/s


In [14]:
# Cross-tradition comparison preview (full table built in 05_evaluation)
csv_files = {
    "Western Classical": "eda_maestro_stats.csv",
    "Turkish Makam"    : "eda_turkish_stats.csv",
    "Irish Folk"       : "eda_irish_stats.csv",
    "Hindustani"       : "eda_hindustani_stats.csv",
    "Carnatic"         : "eda_carnatic_stats.csv",
}

rows = []
for tradition, fname in csv_files.items():
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        d = pd.read_csv(fpath)
        rows.append({
            "Tradition"      : tradition,
            "n"              : len(d),
            "Duration (min)" : f"{d['duration_s'].mean()/60:.1f}",
            "Note density"   : f"{d['note_density'].mean():.2f}",
            "Pitch range"    : f"{d['pitch_range'].mean():.1f}",
            "PC entropy"     : f"{d['pc_entropy'].mean():.3f}",
        })

if rows:
    comparison = pd.DataFrame(rows)
    comparison.to_csv(RESULTS_DIR / "eda_cross_tradition_summary.csv", index=False)
    print("\n=== Cross-Tradition EDA Summary ===")
    print(comparison.to_string(index=False))
    print("\nSaved → results/eda_cross_tradition_summary.csv")
else:
    print("Run remaining EDA notebooks to populate the cross-tradition table.")


=== Cross-Tradition EDA Summary ===
        Tradition   n Duration (min) Note density Pitch range PC entropy
Western Classical 150            9.9        10.79        68.2      3.356
    Turkish Makam 200            2.2         2.03        15.8      2.719

Saved → results/eda_cross_tradition_summary.csv
